In [1]:
import arcgis
import time
from arcgis.gis import GIS
from arcgis.gis import Item
from arcgis.apps.storymap import StoryMap

from typing import Set  # Import Set from typing
import re, json, csv

import pandas as pd
import os
import logging
import requests

# Set Pandas dataframe display options
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns',1000)

In [3]:
agoNotebook = False
# Print the version of the arcgis module
print(f"Running ArcGIS API for Python version: {arcgis.__version__}")

# Define the GIS
if agoNotebook == False:
    import keyring
    service_name = "system" # Use the default local credential store
    success = False # Set initial state

    # Ask for the username
    while success == False:
        username_for_keyring = input("Enter your ArcGIS Online username:") # If you are using VS Code, the text input dialog box appears at the top of the window
        # Get the credential object
        credential = keyring.get_credential(service_name, username_for_keyring)
        # Check if the username is in the credential store
        if credential is None:
            print(f"'{username_for_keyring}' is not in the local system's credential store. Try another username.")
        # Retrieve the password, login and set the GIS portal
        else:
            password_from_keyring = keyring.get_password("system", username_for_keyring)
            portal_url = 'https://www.arcgis.com'  
            gis = GIS(portal_url, username=username_for_keyring, password=password_from_keyring)
            success = True
            # Print a success message with username and user's organization role
            print("Successfully logged in as: " + gis.properties.user.username, "(role: " + gis.properties.user.role + ")")
else:
    gis = GIS("home")

Running ArcGIS API for Python version: 2.4.2
Successfully logged in as: dasbury_storymaps (role: org_admin)


In [4]:
classic_maptour_id = "20fd39888a444629bc8e40d9b6ac38cc"
classic_maptour_webmap = ""
classic_maptour_featureCollection = ""
classic_maptour_featureSet = ""

In [ ]:
# Retrieve the JSON data for the classic MapTour item
classic_item = gis.content.get(classic_maptour_id)
classic_item_json = classic_item.get_data()
import pprint
pprint.pprint(classic_item_json)  # Optional: inspect structure

# Find the webmap ID referenced in the item JSON (usually in 'values' > 'webmap')
webmap_id = None
if 'values' in classic_item_json and 'webmap' in classic_item_json['values']:
    webmap_id = classic_item_json['values']['webmap']
    print(f"Found webmap ID: {webmap_id}")
else:
    print("Webmap ID not found in item JSON.")

# Download the webmap's JSON data
classic_maptour_webmap = None
if webmap_id:
    webmap_item = gis.content.get(webmap_id)
    classic_maptour_webmap = webmap_item.get_data()
    pprint.pprint(classic_maptour_webmap)  # Optional: inspect structure
else:
    print("Cannot retrieve webmap JSON without webmap ID.")

# Parse the webmap JSON to get the featureCollection and featureSet
classic_maptour_featureCollection = None
classic_maptour_featureSet = None
if classic_maptour_webmap:
    # Look for operationalLayers with type 'Feature Layer' or 'featureCollection'
    layers = classic_maptour_webmap.get('operationalLayers', [])
    for layer in layers:
        # Check for featureCollection
        if 'featureCollection' in layer:
            classic_maptour_featureCollection = layer['featureCollection']
            print("Found featureCollection in webmap.")
            # Check for featureSet inside featureCollection
            if 'layers' in classic_maptour_featureCollection:
                for fc_layer in classic_maptour_featureCollection['layers']:
                    if 'featureSet' in fc_layer:
                        classic_maptour_featureSet = fc_layer['featureSet']
                        print("Found featureSet in featureCollection.")
                        break
            break
    if not classic_maptour_featureCollection:
        print("No featureCollection found in webmap.")
    if not classic_maptour_featureSet:
        print("No featureSet found in featureCollection.")
else:
    print("Webmap JSON not loaded.")

In [ ]:
# for i, feature in enumerate(classic_maptour_featureSet):